# POPSIGN — Extraction Pilot (single video / small batch)

**What this notebook does.** A fast sanity check of MediaPipe Holistic quality
on POPSIGN **without downloading a whole train/test dataset** (170-200GB per
train part). It lists a few candidate files straight from Kaggle (metadata
only, no download), pulls just `N_PILOT_VIDEOS` of them individually via
`kagglehub.dataset_download(handle, path=<file>)`, extracts landmarks
in-process, and replays the result with the skeleton overlay drawn on top —
the only way to catch a detector that's confidently tracking the *wrong*
region, which no numeric detection-rate proxy can tell apart from "no hands
detected".

Split out from `popsign.0.dataset.extraction.ipynb` (2026-09-13): that notebook
now owns the real staged bulk extraction (one full dataset downloaded,
extracted and deleted at a time); this one never downloads a full dataset at
all, so it is safe and cheap to run before, during, or between those stages.

## Why per-file download, not the bulk dataset download

`kagglehub.dataset_download(handle)` with no `path` always fetches the
**entire** dataset archive — there is no "just the first video" bulk mode.
Passing `path=<relative file path>` instead makes kagglehub fetch only that
one file (verified 2026-09-13: pulling one 3.3MB clip left only that file, not
the 174GB `train-a-e` archive, in the kagglehub cache). `path` must be an exact
file path — a directory prefix 404s — so this notebook lists file names first
(the Kaggle API's `dataset_list_files`, metadata only, ~200 files/request) and
downloads only the ones it picked.

**This does not scale to bulk extraction** — measured throughput is ~5s/file
(bandwidth-limited per request on this connection), so a train part's ~43K
files would take the better part of two days one-by-one vs. a single
continuous archive transfer. That's exactly why the main extraction notebook
still bulk-downloads each full part instead of doing this at scale.

## Output

Throwaway only: sampled videos go to the kagglehub default cache
(`~/.cache/kagglehub/datasets/...`) and are deleted by the last cell; npz +
overlay GIF go to `data/temp/popsign_pilot_single/` (repo temp policy —
nothing here survives the notebook that wrote it). Nothing is written to
`data/cache/` or `data/raw/`.

## Design decisions

- The Kaggle API's file listing is **not a uniform sample of the whole
  dataset** — it's whichever files the first `LIST_PAGES` pages return (a few
  hundred, in whatever order the API lists them). Fine for "does the detector
  look right", not a claim about dataset-wide statistics (that's what the
  confidence-tuning notebook's seeded 100-video sample is for).
- Extraction runs `n_workers=1` (in-process, no worker pool) — the only mode
  that can run directly in a Jupyter kernel on Windows (see the main
  extraction notebook's "Execution model" section); fine for a handful of
  videos.
- The overlay replay is an **animated GIF, not an MP4** — this machine's
  OpenCV/FFmpeg build has no working H.264 encoder, and its fallback codec
  isn't something browsers decode inline (`IPython.display.Video` on it
  renders a blank 0:00 placeholder). See §4 below and
  `sb.extract.overlay.render_gif`'s docstring.
- Cleanup deletes only the **specific files this notebook downloaded**, not
  the dataset's whole kagglehub cache — a bulk extraction stage could be
  downloading the same dataset concurrently, and this must never delete out
  from under it.

In [ ]:
# ============================================================
# Setup — imports and pilot config
# ============================================================
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display
import kagglehub
from kaggle.api.kaggle_api_extended import KaggleApi

import sb.extract.holistic as ex
from sb.extract.overlay import render_gif
from sb.core.paths import DATASET_IDS, TEMP_DIR, cleanup_temp

SEED = 123                    # pilot sampling seed (independent of the extraction notebook's SEED=42)
PILOT_TRAIN_INDEX = 0         # which DATASET_IDS["TRAIN"] part to sample from (0 = train-a-e)
PILOT_HANDLE = DATASET_IDS["TRAIN"][PILOT_TRAIN_INDEX]
N_PILOT_VIDEOS = 1            # how many videos to download + extract this run
N_OVERLAY_TO_SHOW = 1         # how many of them to render + display inline (keeps cell output small)
LIST_PAGES = 3                # Kaggle file-listing pages to draw the sample pool from (~200 files/page)
PAGE_SIZE = 200                # Kaggle API max page size

PILOT_DIR = TEMP_DIR / "popsign_pilot_single"  # throwaway (data/temp policy)
PILOT_DIR.mkdir(parents=True, exist_ok=True)

api = KaggleApi()
api.authenticate()

print(f"pilot handle : {PILOT_HANDLE}")
print(f"n videos     : {N_PILOT_VIDEOS} (showing overlay for {N_OVERLAY_TO_SHOW})")
print(f"pilot dir    : {PILOT_DIR} (temp — deleted at the end)")

## 1. List candidate files — metadata only, no download

`KaggleApi.dataset_list_files` returns file names + sizes without downloading
anything. `LIST_PAGES × PAGE_SIZE` files (~600 by default) form the sample
pool `N_PILOT_VIDEOS` is drawn from — not the whole dataset (see the title
cell), just enough to pick a few videos at random without paging through the
entire ~43K-file listing first.

In [2]:
# ============================================================
# List a sample pool of files (metadata only) + pick N at random
# ============================================================
_pool = []
_token = None
for _ in range(LIST_PAGES):
    _res = api.dataset_list_files(PILOT_HANDLE, page_token=_token, page_size=PAGE_SIZE)
    _pool.extend(_res.dataset_files)
    _token = _res.next_page_token
    if not _token:
        break
print(f"listed {len(_pool)} candidate files ({LIST_PAGES} page(s), not the whole dataset)")

rng = np.random.default_rng(SEED)
picked = rng.choice(_pool, size=min(N_PILOT_VIDEOS, len(_pool)), replace=False)

# layout is <sign>/<sign>/<id>.mp4 (same as the main extraction notebook's manifest)
pilot_df = pd.DataFrame([
    {"rel_path": f.name, "label": Path(f.name).parts[-2], "id": Path(f.name).stem,
     "total_bytes": f.total_bytes}
    for f in picked
])
display(pilot_df)

listed 600 candidate files (3 page(s), not the whole dataset)


,rel_path,label,id,total_bytes
0,after/after/4a.6001-after-2023_01_04_12_47_40....,after,4a.6001-after-2023_01_04_12_47_40.140-0,3797326


## 2. Download only the sampled file(s)

One `kagglehub.dataset_download(handle, path=<file>)` call per sampled video —
each pulls only that file (confirmed 2026-09-13: cache footprint after one call
was 3.4MB, not the dataset's 174GB). Goes to kagglehub's default cache
(`~/.cache/kagglehub/datasets/...`), same as every other download in this
repo — no external drives, no `output_dir` override.

In [3]:
# ============================================================
# Download the sampled files individually (path= -> single-file fetch)
# ============================================================
downloaded_paths = []
for _row in pilot_df.itertuples(index=False):
    _local = Path(kagglehub.dataset_download(PILOT_HANDLE, path=_row.rel_path))
    downloaded_paths.append(_local)
    print(f"  {_row.rel_path} -> {_local} ({_local.stat().st_size / 1e6:.1f} MB)")

pilot_df = pilot_df.assign(file_path=[str(p) for p in downloaded_paths])

  after/after/4a.6001-after-2023_01_04_12_47_40.140-0.mp4 -> C:\Users\user2\.cache\kagglehub\datasets\mrgeislinger\popsign-asl-v1-0-game-train-a-e-signs\versions\1\after\after\4a.6001-after-2023_01_04_12_47_40.140-0.mp4 (3.8 MB)


## 3. Extract landmarks — in-process (`n_workers=1`, no worker pool)

In [ ]:
# ============================================================
# Extract landmarks for the sampled videos (in-process — safe in a kernel)
# ============================================================
pilot_summary = ex.extract_dataset(
    pilot_df[["file_path", "label", "id"]], split="pilot", out_root=PILOT_DIR,
    n_workers=1, progress=False)
print(pilot_summary)

pilot_npz = {}
for _row in pilot_df.itertuples(index=False):
    _npz_path = PILOT_DIR / "pilot" / _row.label / f"{_row.id}.npz"
    # load eagerly + close the file immediately (`with`) — np.load() otherwise
    # keeps the npz's file handle open for as long as the NpzFile object is
    # referenced, and Windows refuses to delete a file that's still open,
    # which made the final cleanup cell's shutil.rmtree fail with
    # PermissionError once pilot_npz held these across cells (found 2026-09-13)
    with np.load(_npz_path) as _zf:
        _lm = _zf["landmarks"]
        _fps = float(_zf["fps"])
    assert _lm.ndim == 3 and _lm.shape[1:] == (543, 3), f"bad shape: {_lm.shape}"
    pilot_npz[_row.id] = {"landmarks": _lm, "fps": _fps}
    print(f"  {_row.id}: {_lm.shape} {_lm.dtype} · fps {_fps:.1f} · "
          f"NaN {float(np.isnan(_lm.astype(np.float32)).mean()):.1%}")

## 4. Replay with the skeleton overlay

Renders `N_OVERLAY_TO_SHOW` of the extracted videos (default 1, to keep cell
output small even if `N_PILOT_VIDEOS` is a larger batch) as an **animated
GIF**, not an MP4. This machine's OpenCV/FFmpeg build has no working H.264
encoder — `avc1`/`H264` fourccs silently fall back to a non-decodable stream
(missing `libopenh264` DLL), and the working fallback codec (`mp4v`, MPEG-4
Part 2) is not something browsers decode inline: `IPython.display.Video` on an
`mp4v` file renders a blank 0:00 placeholder. GIF has no codec dependency and
displays inline anywhere `IPython.display.Image` does — see
`sb.extract.overlay.render_gif`'s docstring for the size tuning that keeps it
to a few MB.

In [ ]:
# ============================================================
# Overlay replay — skeleton drawn on every frame (animated GIF, see markdown)
# ============================================================
for _row in pilot_df.head(N_OVERLAY_TO_SHOW).itertuples(index=False):
    _z = pilot_npz[_row.id]
    _overlay_path = PILOT_DIR / f"{_row.id}_overlay.gif"
    render_gif(Path(_row.file_path), _z["landmarks"], _overlay_path, fps=float(_z["fps"]))
    print(f"{_row.id}: wrote {_overlay_path} ({_overlay_path.stat().st_size / 1e6:.1f} MB)")
    display(Image(filename=str(_overlay_path)))

## 5. Cleanup — delete only the file(s) this notebook downloaded

Deletes exactly the sampled video files from the kagglehub cache (not the
whole dataset's cache — a bulk extraction stage could be mid-download of the
same dataset) and the throwaway temp tree. Watch the replay above first;
re-run the notebook top-to-bottom to sample a fresh video.

In [1]:
# ============================================================
# Cleanup — delete only the downloaded file(s), then the temp tree
# ============================================================
for _p in downloaded_paths:
    if _p.exists():
        _p.unlink()
        print(f"deleted {_p}")
cleanup_temp()
print(f"deleted temp tree ({TEMP_DIR}) — nothing here was meant to persist")

NameError: name 'downloaded_paths' is not defined